In [6]:
import pandas as pd

# Show every column and a few more rows when printing tables
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

# The data lives one folder up from this notebook (.. means "go up one folder")
df = pd.read_csv('../datasets/diabetic_data.csv', na_values='?', low_memory=False)
mapping = pd.read_csv('../datasets/IDS_mapping.csv')

print('Raw data:', df.shape, '(expect 101,766 rows and 50 columns)')
print('Mapping file:', mapping.shape)
print('Primary diagnosis (diag_1) missing:', df['diag_1'].isna().sum(), '(expect 21)')

Raw data: (101766, 50) (expect 101,766 rows and 50 columns)
Mapping file: (67, 2)
Primary diagnosis (diag_1) missing: 21 (expect 21)


In [5]:
import os
print('Running from:', os.getcwd())
print('Folders and files here:', os.listdir('.'))

Running from: c:\Users\patri\OneDrive\Documents\Data Analytics\Projects\healthcare\notebooks
Folders and files here: ['diabetic_healthcare_cleaned.ipynb']


In [145]:
# PART 1: INSPECT THE RAW DATA

# ---------- 1a) The first rows and the size ----------
print('Rows and columns:', df.shape, '\n')
df.head()

# ---------- 1b) Every column, its type, and how many values are filled in ----------
df.info()

# ---------- 1c) Which columns have missing values, and how many ----------
missing = df.isna().sum()                      # blanks per column
missing = missing[missing > 0]                 # keep only columns that have any
missing = missing.sort_values(ascending=False) # most missing first

print(pd.DataFrame({'missing': missing,
                    'percent': (missing / len(df) * 100).round(1)}))

# ---------- 1d) The original readmission column ----------
# NO = never readmitted, >30 = readmitted after 30 days, <30 = readmitted within 30 days
print(df['readmitted'].value_counts())
print()
print('Raw 30-day readmission rate:', round((df['readmitted'] == '<30').mean() * 100, 2), '%')

# ---------- 1e) Smallest, middle and largest value of each number column ----------
df.describe().T[['min', '50%', 'max']]

Rows and columns: (101766, 50) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            

,min,50%,max
encounter_id,12522.0,152388987.0,443867222.0
patient_nbr,135.0,45505143.0,189502619.0
admission_type_id,1.0,1.0,8.0
discharge_disposition_id,1.0,1.0,28.0
admission_source_id,1.0,7.0,25.0
time_in_hospital,1.0,4.0,14.0
num_lab_procedures,1.0,44.0,132.0
num_procedures,0.0,1.0,6.0
num_medications,1.0,15.0,81.0
number_outpatient,0.0,0.0,42.0


In [146]:
# PART 2: CLEANING

# ---------- Step 1: Drop columns that are mostly empty ----------
# weight (97% blank) and payer_code (40% blank) are too empty to use
df = df.drop(columns=['weight', 'payer_code'])
print('Step 1 | columns left:', df.shape[1], '(expect 48)')

# ---------- Step 2: Fill blanks with a label that means something ----------
# A blank A1C or glucose result means the test was not ordered.
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not Tested')
df['A1Cresult'] = df['A1Cresult'].fillna('Not Tested')

# Same idea for the other columns with blanks
df['medical_specialty'] = df['medical_specialty'].fillna('Not Specified')
df['race'] = df['race'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('None')    # no second diagnosis recorded
df['diag_3'] = df['diag_3'].fillna('None')    # no third diagnosis recorded

# Checkpoint: which columns still have blanks?
blanks_left = df.isna().sum()
print('\nStep 2 | blanks left:')
print(blanks_left[blanks_left > 0])

Step 1 | columns left: 48 (expect 48)

Step 2 | blanks left:
diag_1    21
dtype: int64


In [147]:
# ---------- Step 3: Drop patients with no primary diagnosis ----------
# Only 21 patients. Without diag_1 we can't say why they were in the hospital.
df = df.dropna(subset=['diag_1'])
print('Step 3 | after dropping missing diag_1:', df.shape,
      '| blanks left:', df.isna().sum().sum(), '(expect 101,745 rows, 0 blanks)')

# ---------- Step 4: Remove patients who died or went to hospice ----------
# Discharge codes: 11 = expired, 13 and 14 = hospice, 19/20/21 = expired (other settings)
exclude_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(exclude_ids)]
print('Step 4 | after removing expired/hospice:', df.shape, '(expect 99,323 rows)')

Step 3 | after dropping missing diag_1: (101745, 48) | blanks left: 0 (expect 101,745 rows, 0 blanks)
Step 4 | after removing expired/hospice: (99323, 48) (expect 99,323 rows)


In [148]:
# ---------- Step 5: Keep only each patient's FIRST visit ----------
# Some patients appear many times. Counting the same person again and again would give them too much weight
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
print('Step 5 | after first visit per patient:', df.shape, '(expect 69,982 rows)')

# ---------- Step 6: Create the target column ----------
# We only care about readmission within 30 days: 1 = yes, 0 = no.
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)
print('Step 6 | target counts:', df['readmitted_30d'].value_counts().to_dict(),
      '| 30-day rate:', round(df['readmitted_30d'].mean() * 100, 2), '% (expect 8.98%)')

Step 5 | after first visit per patient: (69982, 48) (expect 69,982 rows)
Step 6 | target counts: {0: 63699, 1: 6283} | 30-day rate: 8.98 % (expect 8.98%)


In [149]:
# PART 3: SANITY CHECKS

# Each patient should now appear exactly once 
print('Each patient appears once:', df['patient_nbr'].is_unique)

# Final size 
print('Final shape:', df.shape)

# Readmitted patients and overall rate 
print('Readmitted:', df['readmitted_30d'].sum(), 'of', len(df))
print('Overall 30-day readmission rate:', round(df['readmitted_30d'].mean() * 100, 2), '%')

# Gender: a few "Unknown/Invalid" rows exist; they are too few to matter
print()
print(df['gender'].value_counts())

Each patient appears once: True
Final shape: (69982, 49)
Readmitted: 6283 of 69982
Overall 30-day readmission rate: 8.98 %

gender
Female             37237
Male               32742
Unknown/Invalid        3
Name: count, dtype: int64


In [152]:
# ---------- Step 7: Group the diagnosis codes into broad categories ----------
# diag_1 holds ICD-9 codes (694 different ones). They are grouped into 9 categories, using the grouping from the original study.

def group_diagnosis(code):
    # Codes starting with V or E are special-case codes that can't be turned
    # into numbers, so they go straight to Other
    if code.startswith(('V', 'E')):
        return 'Other'

    # Turn the text code into a whole number: '250.7' -> 250, '428' -> 428
    base = int(float(code))

    # Check which range the code falls in; the first match wins
    if base == 250:
        return 'Diabetes'
    elif 390 <= base <= 459 or base == 785:
        return 'Circulatory'
    elif 460 <= base <= 519 or base == 786:
        return 'Respiratory'
    elif 520 <= base <= 579 or base == 787:
        return 'Digestive'
    elif 800 <= base <= 999:
        return 'Injury'
    elif 710 <= base <= 739:
        return 'Musculoskeletal'
    elif 580 <= base <= 629 or base == 788:
        return 'Genitourinary'
    elif 140 <= base <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# Apply it to every patient and save the result as a new column
df['diag_category'] = df['diag_1'].apply(group_diagnosis)

print('Step 7 | patients by diagnosis category:')
print(df['diag_category'].value_counts())
print('Total:', df['diag_category'].value_counts().sum())

Step 7 | patients by diagnosis category:
diag_category
Circulatory        21392
Other              12124
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4696
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64
Total: 69982


In [7]:
# SAVE THE CLEAN DATA

df.to_csv('../datasets/diabetic_clean.csv', index=False)
print('Saved ../datasets/diabetic_clean.csv with shape', df.shape, '(expect 69,982 rows and 50 columns)')

Saved ../datasets/diabetic_clean.csv with shape (101766, 50) (expect 69,982 rows and 50 columns)
